# SQLite, through a hostcall

The Lab has real SQLite — compiled to WebAssembly and vendored — and a
program reaches it the way it reaches the clock: by **asking its host**.

That distinction is the whole design and it is worth getting straight
before any SQL happens. There is no `sqlite` library inside the sandbox;
`notebooks/sql.ipynb` builds a small engine in Lua precisely because a
sealed program has no database in it. What exists here is a *connector*: a
program granted `host:sql/query` sends a message and rows come back. A
program not granted it cannot reach the database at all, which is what
makes the grant worth writing down.

This is also the production shape. `host/dhost_sql.c` answers the same two
calls over the system SQLite, so a program written here moves to the C
host unchanged.


In [ ]:
-- SQL is host-side, so this needs the swarm layer (v5.5.1_build5+).
if type(swarm) ~= "table" then
  print("SKIP -- needs the Lab swarm runner (v5.5.1_build5 or newer)")
  df_sql_ready = false
else
  df_sql_ready = true
  print("ready")
end


## The shape of a hostcall

`{tok, call, args}` out on `host/calls`, `{tok, status, value}` back on
`host/replies`. The **correlation token is required from the first
prototype**: a program may have several calls outstanding and replies
arrive in whatever order the host answers them, so matching by arrival
order is a bug waiting for concurrency.

The helper below runs a root program and hands back whatever it pushed to
its outbox, so the rest of this notebook is about SQL rather than
plumbing.


In [ ]:
-- Everything below shares this. `mode` and `max_rows` are the connector's
-- configuration, in `host/example.host.lua`'s shape on purpose.
function run_sql(body, config)
  if not df_sql_ready then return {} end
  swarm.stop()
  swarm.start{
    root = [[
      local out     = queue.lookup("outbox")
      local calls   = queue.declare("host/calls",   { capacity = 8, exported = true })
      local replies = queue.declare("host/replies", { capacity = 8 })

      local tok = 0
      function hostcall(name, args)
        tok = tok + 1
        local mine = tok
        queue.push(calls, { tok = mine, call = name, args = args })
        while true do
          local _, reply = queue.wait({ replies })
          if reply.tok == mine then return reply end
        end
      end

      function say(...) queue.push(out, table.concat({ ... }, " ")) end

      -- Report the status and, for a query, the rows.
      --
      -- A third argument of "refused" marks a line whose whole point is
      -- the refusal, so it reads as the pass it is -- and an unexpected
      -- *success* becomes the thing that stands out.
      function show(label, r, expect)
        if expect == "refused" then
          if r.status == "ok" then
            say(label, "->", "UNEXPECTED -- this was supposed to be refused")
          else
            say(label, "->", "refused as designed (" .. r.status .. "):", tostring(r.detail))
          end
        elseif r.status ~= "ok" then
          say(label, "->", r.status .. ":", tostring(r.detail))
        elseif type(r.value) == "table" and r.value.rows then
          say(label, "->", "ok," , tostring(#r.value.rows), "row(s)")
          for _, row in ipairs(r.value.rows) do
            local cs = {}
            for i, c in ipairs(row) do cs[i] = tostring(c) end
            say("   ", table.concat(cs, " | "))
          end
        else
          say(label, "->", "ok")
        end
      end
    ]] .. body,
    caps = { "queue:*", "host:sql/query", "host:sql/exec" },
    budget = { instructions = 20000000, memory_kb = 256 },
    max_instances = 4,
    connectors = { sql = config or { path = "lab.db", mode = "readwrite", max_rows = 64 } },
  }
  swarm.step(200)
  local out = swarm.drain("root", "outbox")
  -- Deliberately NOT stopped: the swarm, and the database it built,
  -- stay up so the Instances panel has something to show -- and
  -- something to download. The swarm.stop() above clears the previous
  -- run, so each cell still starts from an empty database.
  return out
end

function show_sql(body, config)
  for _, line in ipairs(run_sql(body, config)) do print(tostring(line)) end
end
print("run_sql is ready")


## What the Lua engine could not do

`sql.ipynb` refuses joins, subqueries and grouping — honestly, because
writing a query planner is a different notebook. SQLite has all of them,
and the difference is not cosmetic: this is the same engine that will be
underneath in production.


In [ ]:
show_sql([[
  hostcall("sql/exec", { sql = "CREATE TABLE account (id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE)" })
  hostcall("sql/exec", { sql = "CREATE TABLE hit (id INTEGER PRIMARY KEY, owner INTEGER NOT NULL, ms INTEGER)" })
  for _, n in ipairs({ "ada", "annie", "grace" }) do
    hostcall("sql/exec", { sql = "INSERT INTO account (name) VALUES (?)", params = { n } })
  end
  for _, h in ipairs({ {1,10}, {1,25}, {2,40}, {3,5}, {3,15}, {3,30} }) do
    hostcall("sql/exec", { sql = "INSERT INTO hit (owner, ms) VALUES (?, ?)", params = { h[1], h[2] } })
  end

  show("join + group by + order by", hostcall("sql/query", {
    sql = [==[
      SELECT a.name, COUNT(h.id) AS n, SUM(h.ms) AS total
      FROM account a JOIN hit h ON h.owner = a.id
      GROUP BY a.name ORDER BY total DESC
    ]==] }))

  show("a subquery", hostcall("sql/query", {
    sql = "SELECT name FROM account WHERE id IN (SELECT owner FROM hit WHERE ms > ?)",
    params = { 30 } }))

  show("a CTE", hostcall("sql/query", {
    sql = "WITH slow AS (SELECT owner FROM hit WHERE ms >= 25) "
       .. "SELECT COUNT(*) FROM slow" }))
]])


## SQLite's own semantics, not an imitation of them

Constraints fire because SQLite enforces them, and `NULL` compares equal to
nothing — including itself. A hand-written engine has to remember to do
this; a real one cannot forget.


In [ ]:
show_sql([[
  hostcall("sql/exec", { sql = "CREATE TABLE t (id INTEGER PRIMARY KEY, k TEXT NOT NULL UNIQUE, note TEXT)" })
  hostcall("sql/exec", { sql = "INSERT INTO t (k) VALUES ('one')" })
  show("a duplicate key",  hostcall("sql/exec", { sql = "INSERT INTO t (k) VALUES ('one')" }), "refused")
  show("a NULL where NOT NULL", hostcall("sql/exec", { sql = "INSERT INTO t (k) VALUES (NULL)" }), "refused")
  hostcall("sql/exec", { sql = "INSERT INTO t (k, note) VALUES ('two', NULL)" })
  show("note = NULL",   hostcall("sql/query", { sql = "SELECT id FROM t WHERE note = NULL" }))
  show("note IS NULL",  hostcall("sql/query", { sql = "SELECT id FROM t WHERE note IS NULL" }))
]])


## The confinement, which is the weaker half

Here is the part to read before trusting this with anything.

The **contract** is the C host's exactly — same two calls, same shapes,
same read/write split. The **confinement** is not. `host/dhost_sql.c`
earns its confinement from three SQLite primitives no JavaScript driver
exposes: `sqlite3_set_authorizer`, `SQLITE_LIMIT_ATTACHED` and
`sqlite3_stmt_readonly`. Without them the escapes are gated on the
statement's *text*, which is a floor rather than a target.

So: build to the contract so your guest cannot tell the two apart, and do
not point this at a database that matters. Production is the C host.

Four gates, each visible below.


In [ ]:
show_sql([[
  hostcall("sql/exec", { sql = "CREATE TABLE t (id INTEGER PRIMARY KEY, a TEXT)" })

  -- 1. Transactions, ATTACH and PRAGMA are host state held against a
  -- guest, and the v1 encoding has nowhere to put a handle spanning calls.
  show("BEGIN",   hostcall("sql/exec", { sql = "BEGIN" }), "refused")
  show("PRAGMA",  hostcall("sql/exec", { sql = "PRAGMA journal_mode = WAL" }), "refused")
  show("ATTACH",  hostcall("sql/exec", { sql = "ATTACH DATABASE 'other.db' AS other" }), "refused")

  -- 2. One statement per call. The drivers prepare the first and ignore
  -- the rest, so a second would ride in unauthorised AND unrun -- worse
  -- than either running it or refusing it. SQLite's own parser decides
  -- where a statement ends, so a `;` inside a literal is not a separator.
  show("two statements", hostcall("sql/exec", {
    sql = "INSERT INTO t (a) VALUES ('x'); DROP TABLE t" }), "refused")
  show("a ; inside a literal", hostcall("sql/exec", {
    sql = "INSERT INTO t (a) VALUES ('x;y')" }))
  show("the table survived", hostcall("sql/query", { sql = "SELECT COUNT(*) FROM t" }))

  -- 3. The parameter count must match exactly. Too few silently NULL-binds
  -- the rest, which is the same class of quiet wrongness as a truncated
  -- result.
  show("too few params", hostcall("sql/exec", {
    sql = "INSERT INTO t (id, a) VALUES (?, ?)", params = { 99 } }), "refused")
]])


### The row cap refuses rather than truncating

A truncated result is a silent lie: the program gets rows, believes it got
*the* rows, and is wrong. So the cap is an error, and the guest pages with
`LIMIT`/`OFFSET` instead.

It is also checked while stepping a cursor rather than after materialising
— counting a hostile result set *after* building it in memory is a denial
of service the C host does not have.


In [ ]:
show_sql([[
  hostcall("sql/exec", { sql = "CREATE TABLE n (i INTEGER PRIMARY KEY)" })
  for i = 1, 12 do hostcall("sql/exec", { sql = "INSERT INTO n (i) VALUES (?)", params = { i } }) end
  show("all 12, cap is 5", hostcall("sql/query", { sql = "SELECT i FROM n" }), "refused")
  show("paged to 5",       hostcall("sql/query", { sql = "SELECT i FROM n LIMIT 5" }))
]], { path = "lab.db", mode = "readwrite", max_rows = 5 })


### A grant that is read-only leaves `sql/exec` unwired

`mode = "read"` is not a runtime check inside the connector — the write
call is simply never wired, so asking for it is `denied` the way any
ungranted call is. The capability and the configuration agree, which is
the property worth having.


In [ ]:
show_sql([[
  show("a read",  hostcall("sql/query", { sql = "SELECT 1 AS one" }))
  show("a write", hostcall("sql/exec",  { sql = "CREATE TABLE t (id INTEGER)" }), "refused")
]], { path = "lab.db", mode = "read", max_rows = 16 })


## The database is a file

The database lives in memory — a browser tab has no filesystem, so the
`path` above names a database rather than locating one, and it is gone
when the kernel restarts.

It can still leave, and arrive. In the **Instances** panel:

- **Download .sqlite** exports it. What comes out begins `SQLite format 3`,
  because SQLite serialised it — `sqlite3`, a GUI, or another Lab session
  will open it.
- **Open .sqlite…**, beside the program picker, goes the other way. The
  file is staged and opened when you press **Start**, because the database
  is built when the swarm is. A file that is not a database is refused
  when you choose it, by name.

Those are buttons, so this notebook cannot press them — a cell drives the
kernel, not the page. What it *can* do is show that the bytes survive a
round trip, which is the property those buttons rest on: run the cell
below, then open the panel and download what it made.


In [ ]:
show_sql([[
  hostcall("sql/exec", { sql = "CREATE TABLE note (id INTEGER PRIMARY KEY, body TEXT)" })
  hostcall("sql/exec", { sql = "INSERT INTO note (body) VALUES (?)",
                         params = { "written in a cell, readable in sqlite3" } })
  show("stored", hostcall("sql/query", { sql = "SELECT body FROM note" }))
  say("")
  say("Open the Instances panel and press Download .sqlite to take this away.")
]])


## Where this goes

`doc/Host.md`'s acceptance test is that **a guest must not be able to tell
two hosts apart**. For the contract that holds exactly, and now for the
engine too: a query that runs here runs on the C host, and a constraint
that fires here fires there.

What does not hold is the confinement, and the Lab says so rather than
letting you find out — in the panel, in the README, and in the refusals
above. Prototype the schema here; run it where the authorizer is.

Related notebooks: **A swarm, from a cell** for the layer this rides on,
and **Building SQL in Lua** for what a program does when it has no host to
ask.
